# T1 · Picking the batches

**Tiny Transformer, Unit 2, module T1.** Every training step of the recorded run read 64 windows of the training text. Where
did they come from? Here you re-draw all 100,000 steps' window starts from the run's own seed, exactly as `train.py` drew them,
and count them: which places were never drawn, which characters were never once a right answer, what the seed decided, and how
many times over the run read its text.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T1-training-batches.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T1-training-batches.ipynb`.

**Computed here:** every step's 64 starts, the counts, the checks' fixed windows, and the model's starting numbers. **Checked
against the run:** its recorded first batch (the same 64 starts) and the course's counts, which the notebook asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import hashlib, os, urllib.request
import torch

print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole (making the model draws its starting numbers); then Unit 2's teaching functions, `code/training_steps.py`, copied in
whole: T1's are the first five.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os
import re

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best


# ---------------------------------------------------------------------------------------------------- T7 · the curve

def share_of_fall(checks, step):
    """How much of its whole fall a run's held-back check had made at `step`, read from the check at that step: from the first
    check down to the best one, as a share (0: none yet, 1: all of it). `checks` is a log of [step, held-back score] pairs."""
    first, best = checks[0][1], min(score for _, score in checks)
    now = dict(checks)[step]                                      # the check at that step
    return (first - now) / (first - best)


def worse_than_before(checks):
    """The steps whose check scored worse (higher) than the check just before it, in a log of [step, held-back score] pairs."""
    return [step for (_, before), (step, score) in zip(checks, checks[1:]) if score > before]


def gaps(log):
    """The held-back check minus the training check, at every check: [step, gap] pairs, from a log of [step, training score,
    held-back score]. Both are scored at the same moment, on the same numbers: only the text differs."""
    return [[step, val - train] for step, train, val in log]


def write(model, tok, n=1000, seed=42, temperature=1.0):
    """What a model writes from a new line: n characters, picked with its own generator, as the course's writing samples are
    made (training/prompter/unit2.py). The same start, seed and temperature every time, so only the numbers differ."""
    model.eval()
    with torch.no_grad():
        out = model.generate(torch.tensor([tok.encode("\n")]), n, temperature, torch.Generator().manual_seed(seed))
    return tok.decode(out[0, 1:].tolist())                        # the new line it started from left out


def word_share(text, training_text):
    """The share of a text's words found in the training text, and the words found nowhere in it. A word is a run of letters
    and apostrophes, read in lower case. Found means spelt somewhere in the training text: it says nothing about sense."""
    known = set(re.findall(r"[a-z']+", training_text.lower()))   # every word the training text spells
    words = re.findall(r"[a-z']+", text.lower())
    found = sum(w in known for w in words)
    return found / len(words), sorted({w for w in words if w not in known})

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the last 10% is held back
n = len(train_ids)
print(f"training text {n:,} characters · possible starts {n - 32 - 1:,}")
assert n == 1003854 and n - 32 - 1 == 1003821

training text 1,003,854 characters · possible starts 1,003,821


# Section 1 · Picking windows

One step's windows, followed back into the training text: where they start, and which characters they ask about. Each chapter
below follows its page: Picture it, Watch it in our run, then Read and run the code.

## T1.1 · 64 random places

*Where do each step's 64 windows start?*

**Predict first** (the page asks it too, before its clip): over the whole run, 6,400,000 windows were drawn from 1,003,821 possible starting places: about 6 each.
Were some places never drawn: yes, about 1,700 of them; no, about 6 draws each reaches every
place; or yes, about 40,000, 4 in every 100, as at 3 rolls?

### T1.1.1 · Picture it

M6's batch took 64 windows from random places. Does picking at random reach every place? Picture 10 made-up places and a
10-sided die: each roll picks one place. Roll once per place, and about 35 places in 100 get none; twice per place, about 12;
three times, about 4. More rolls shrink the gaps: fewer and fewer places get none. (The page's rolls are made up to show the
idea.)

### T1.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

This is the training text: 1,003,821 places where the run can start a window. Each step drew 64 of them at
random. Over the run's 100,000 steps, 6,400,000 draws: most places were drawn 4 to 9 times; 1,726 were never drawn,
and one was drawn 22 times. Nothing spreads the draws out: each one is a fresh random pick, so some places pile up
and some are missed.

### T1.1.3 · Read and run the code

`run_starts` in `code/training_steps.py` (copied in above) replays the run's draws. It sets the seed, 1337, makes the model first
(its starting numbers take the stream's first draws, as in `train.py`), then calls `torch.randint` every step: 64 whole numbers
below the number of places. Stacked, the steps make one table, [100,000×64]. Then `times_drawn` counts each place's draws with
`torch.bincount`. It takes a second or two.

In [7]:
run = run_starts(n, make_model=lambda: Prompter(cfg))      # [100000×64]  every step's starts, from seed 1337
drawn = times_drawn(run, n)                                # [1003821]   how many times each place was drawn
print('draws', run.numel(), ' places', len(drawn), ' never drawn', int((drawn == 0).sum()), ' most draws', int(drawn.max()))
print('step 1, its first 8 starts:', run[0, :8].tolist())
assert run[0].tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307, 510490, 928356, 614059, 30062, 737496, 407796, 59365, 998494, 864171, 929951, 942896, 637372, 408276, 289366, 811280, 165163, 748492, 428999, 260643, 560439, 371836, 390719, 776246, 678592, 338694, 316420, 711307, 461584, 554475, 251253, 546994, 358001, 64331, 976218, 589122, 206049, 243936, 44076, 936280, 819204, 162295, 566156, 935951, 895342, 457178, 537830, 274346, 823229, 14080, 899619, 528867, 495983, 90293, 96640, 21128, 975278]   # the recorded run's first batch
assert (run.numel(), len(drawn)) == (6400000, 1003821)
assert (int((drawn == 0).sum()), int(drawn.max())) == (1726, 22)
assert torch.bincount(drawn).tolist() == [1726, 10795, 34854, 73520, 117506, 150285, 159252, 145774, 115864, 81842, 52247, 30108, 16238, 7771, 3555, 1522, 605, 229, 89, 26, 12, 0, 1]   # places by times drawn: 0, 1, 2 … 22

draws 6400000  places 1003821  never drawn 1726  most draws 22
step 1, its first 8 starts: [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307]


### ✋ Your turn

How many places were drawn exactly 6 times, about the average? Count them in `drawn` and put the number in `answer`.

In [8]:
answer = None   # TODO: (drawn == 6).sum(), then int(...)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((drawn == 6).sum()), 'compare drawn with 6, then sum the matches: int((drawn == 6).sum())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
answer = int((drawn == 6).sum())
print(f"{answer:,} places drawn exactly 6 times, of {len(drawn):,}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((drawn == 6).sum()), 'compare drawn with 6, then sum the matches: int((drawn == 6).sum())'
    print("✓ right:", answer)

159,252 places drawn exactly 6 times, of 1,003,821
✓ right: 159252


## T1.2 · Is every character asked?

*Which characters of the text are asked about?*

**Predict first** (the page asks it too, before its clip): 1,726 starting places were never drawn. How many characters of the training text were never once a right answer:
only 2, one at each end; about 1,726, one for each place never drawn; or only 1, the very first, which nothing
comes before?

### T1.2.1 · Picture it

A window's right answers are the characters after its starting place, one per position. Picture a made-up text of 12
characters, and windows of 4: a window from place 2 asks about characters 3, 4, 5 and 6. Under each character, count the windows
that ask about it. A character inside the text sits under 4 places, one per position in a window; near the ends, fewer. (The page's text is made up to show the idea.)

### T1.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's counts: how many times each character of the training text was a right answer. At the front, the first
character was asked 0 times, the next 12, then 21: more windows can reach each one. At the far end,
the counts fall the same way, to 5, then 0 for the last character. The last window that fits would start at
`torch.randint`'s limit, and `randint` never draws its limit. Between the ends, no count is 0. A place never drawn leaves no
character out: 32 places cover each one.

### T1.2.3 · Read and run the code

`times_asked` turns the draws into a count for every character, with no loop over windows. Each place adds its number of draws
where its answers begin, one character after it, and takes them away again just after its last answer, the one 32 characters
on. A running total, the cumulative sum (`cumsum`), then gives every character's count.

In [10]:
asked = times_asked(run, n)                                # [1003854]  how many times each character was a right answer
print('the first 5:', asked[:5].tolist(), ' the last 5:', asked[-5:].tolist())
print('never asked:', torch.nonzero(asked == 0).flatten().tolist(), ' of', n, 'characters')
inner = asked[32 + 1:n - 32 - 1]                               # every character a whole 32 places can reach
print('between the ends, any zeros?', bool((inner == 0).any()))
assert torch.nonzero(asked == 0).flatten().tolist() == [0, n - 1]       # only the first and the last
assert asked[:34].tolist() == [0, 12, 21, 26, 31, 39, 44, 49, 54, 59, 69, 75, 82, 94, 102, 107, 111, 115, 126, 140, 149, 155, 160, 168, 173, 180, 184, 190, 196, 201, 208, 218, 221, 216]
assert asked[-34:].tolist() == [192, 191, 186, 180, 169, 162, 157, 151, 147, 144, 139, 134, 128, 122, 116, 107, 104, 100, 94, 89, 83, 75, 72, 67, 54, 43, 39, 31, 24, 20, 16, 12, 5, 0]
assert not bool((inner == 0).any())
assert asked.sum().item() == run.numel() * 32                 # every window asks 32 questions

the first 5: [0, 12, 21, 26, 31]  the last 5: [20, 16, 12, 5, 0]
never asked: [0, 1003853]  of 1003854 characters
between the ends, any zeros? False


### ✋ Your turn

Count step 1 by hand: start from zeros, one per character, and for each of its 64 starts `i`, add 1 to the 32 characters it asks
about, `slow[i + 1:i + 33]`. Put the total of `slow` in `answer`. Is `slow` the same as `times_asked(run[:1], n)`?

In [11]:
slow = torch.zeros(n, dtype=torch.long)
answer = None   # TODO: for i in run[0].tolist(): slow[i + 1:i + 33] += 1; then int(slow.sum())

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 2048 and torch.equal(slow, times_asked(run[:1], n)), '64 windows of 32 answers: loop over run[0], add 1 to slow[i + 1:i + 33], then int(slow.sum())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
slow = torch.zeros(n, dtype=torch.long)
for i in run[0].tolist():
    slow[i + 1:i + 33] += 1
answer = int(slow.sum())
print(answer, torch.equal(slow, times_asked(run[:1], n)))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 2048 and torch.equal(slow, times_asked(run[:1], n)), '64 windows of 32 answers: loop over run[0], add 1 to slow[i + 1:i + 33], then int(slow.sum())'
    print("✓ right:", answer)

2048 True
✓ right: 2048


# Section 2 · The seed and the run

The run's random numbers, in order: what the seed decided, and how often its windows reach each character.

## T1.3 · One seed, two jobs

*What does the seed decide?*

**Predict first** (the page asks it too, before its clip): the run's seed is 1337. With seed 1338 instead, would the checks read different held-back windows: no, the checks draw from a
stream of their own; yes, one seed decides everything random; or yes, but only some of them?

### T1.3.1 · Picture it

A computer's random numbers come from a generator: a stream of numbers that a seed starts. Picture one ribbon of made-up random numbers,
from one seed. The model is made first: its starting numbers take the ribbon's first stretch. Each step's places take what
follows, in order. The same seed gives the same ribbon: the same starting numbers and the same batches, every time. (The page's
ribbon is made up to show the idea.)

### T1.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

These are the run's random numbers, in order. First, `train.py` sets the seed, 1337, once. Then the model is made. Not all its numbers start
random: 8,988 are drawn from the stream; the other 429, its biases and LayerNorm
scales, start at 0 or 1. Then each step draws its 64 places from the same stream. With seed 1338, step 1's first place would
be 102,792, not 160,204. The checks have a stream of their own: a second generator, seed
2024, made once. It picks their windows of held-back text, the validation set, and every check reads the same ones.

### T1.3.3 · Read and run the code

`check_starts` makes the checks' generator, from their own seed, 2024, and every draw names that generator, so the main stream is never
touched. `run_starts` is the main stream: `torch.manual_seed` first, then `make_model`, so the starting numbers take its first
draws. The cell below counts the model's starting numbers, runs one step with seed 1338, and draws the checks' windows twice,
once under each seed.

In [13]:
torch.manual_seed(1337)
model = Prompter(cfg)                                      # its starting numbers: drawn, or set to 0 or 1
kinds = {'drawn': 0, 'at 0': 0, 'at 1': 0}
for p in model.parameters():
    kind = 'at 0' if bool((p == 0).all()) else 'at 1' if bool((p == 1).all()) else 'drawn'
    kinds[kind] += p.numel()
print('starting numbers:', kinds, ' in all', sum(kinds.values()))

other = run_starts(n, steps=1, seed=1338, make_model=lambda: Prompter(cfg))   # [1×64]  seed 1338's first step
print(f"step 1's first place: seed 1337 {run[0, 0].item():,} · seed 1338 {other[0, 0].item():,}")

torch.manual_seed(1337); checks_a = check_starts(n, len(val_ids))
torch.manual_seed(1338); checks_b = check_starts(n, len(val_ids))
print('the checks’ windows: training', tuple(checks_a[0].shape), ' held back', tuple(checks_a[1].shape),
      ' the same under both seeds:', all(torch.equal(a, b) for a, b in zip(checks_a, checks_b)))

torch.manual_seed(1337); before = torch.rand(3)
torch.manual_seed(1337); check_starts(n, len(val_ids)); after = torch.rand(3)
print('the main stream after drawing the checks’ windows: untouched', torch.equal(before, after))
assert kinds == {'drawn': 8988, 'at 0': 345, 'at 1': 84}
assert other[0, :8].tolist() == [102792, 908497, 538125, 385972, 507581, 969893, 863440, 575583] and run[0, :8].tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307]
assert all(torch.equal(a, b) for a, b in zip(checks_a, checks_b)) and torch.equal(before, after)

starting numbers: {'drawn': 8988, 'at 0': 345, 'at 1': 84}  in all 9417
step 1's first place: seed 1337 160,204 · seed 1338 102,792
the checks’ windows: training (100, 64)  held back (100, 64)  the same under both seeds: True
the main stream after drawing the checks’ windows: untouched True


### ✋ Your turn

With seed 1338, what is step 2's first place? Draw 2 steps with `run_starts` (the model made first, as above) and put the place
in `answer`.

In [14]:
answer = None   # TODO: run_starts(n, steps=2, seed=1338, make_model=lambda: Prompter(cfg)), then [1, 0].item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == run_starts(n, steps=2, seed=1338, make_model=lambda: Prompter(cfg))[1, 0].item(), 'row 1 is step 2: run_starts(...)[1, 0].item()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = run_starts(n, steps=2, seed=1338, make_model=lambda: Prompter(cfg))[1, 0].item()
print(f"seed 1338, step 2's first place: {answer:,}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == run_starts(n, steps=2, seed=1338, make_model=lambda: Prompter(cfg))[1, 0].item(), 'row 1 is step 2: run_starts(...)[1, 0].item()'
    print("✓ right:", answer)

seed 1338, step 2's first place: 976,520
✓ right: 976520


## T1.4 · How many times through

*How often does the run read its text?*

**Predict first** (the page asks it too, before its clip): 100,000 steps of 2,048 right answers each. About how many times over is that the training text: about 204 times;
about 6, since each place is drawn about 6 times; or once, since each character is learned once?

### T1.4.1 · Picture it

How much of its text does the run read, and how often? Picture the made-up text of 12 characters again, with 2 windows of 4 each
step. Each step draws 2 places, yet has 8 right answers: 4 in every window. After 3 steps: 6 places drawn, 24 right answers, and a
count under every character. (The page's text is made up to show the idea.)

### T1.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is every character between the ends, by how many times it was a right answer. Most sit near the middle: 204 times on
average, from 137 to 269 between the ends. Each character is asked once for every draw of the
32 places before it: 32 places, 6.38 draws each on average, about 204. The run has no rounds through the text, only steps: no
character waits its turn, so the counts spread.

### T1.4.3 · Read and run the code

`times_through` is one line: the steps times 64 times 32, over the text's length. 100,000 steps of 2,048 is 204,800,000
right answers; over 1,003,854 characters, that is about 204 each. The cell below checks it against `times_asked`: the same average,
and every count between the ends.

In [16]:
through = times_through(n)
mean = asked[1:n - 1].double().mean().item()                # every character but the 2 at the ends
print(f"times through: {through:.2f} · times asked, on average: {mean:.2f}")
print('between the ends: from', int(inner.min()), 'to', int(inner.max()))
assert (int(inner.min()), int(inner.max())) == (137, 269)
assert abs(through - 204.014) < 1e-3 and round(through) == 204
assert abs(mean - 204.014) < 1e-3 and abs(mean - through) < 0.01
assert torch.bincount(inner - 137).tolist() == [2, 1, 0, 0, 0, 1, 2, 1, 2, 2, 2, 1, 7, 14, 15, 25, 30, 46, 58, 70, 95, 100, 149, 173, 239, 278, 306, 434, 566, 673, 869, 1047, 1272, 1495, 1840, 2234, 2608, 2979, 3471, 3973, 4541, 5285, 6204, 6979, 7833, 8685, 9734, 10721, 11732, 13055, 13991, 15529, 16567, 17942, 18904, 20403, 21255, 22367, 23260, 24432, 25196, 25875, 26823, 27352, 27392, 27865, 27807, 27938, 27439, 27628, 26991, 26620, 26098, 25327, 24521, 23642, 22449, 21170, 20458, 19052, 17844, 16736, 15665, 14423, 13514, 12354, 11376, 10523, 9359, 8701, 7771, 6897, 6303, 5460, 4851, 4282, 3868, 3372, 2797, 2412, 2145, 1762, 1516, 1247, 1092, 959, 813, 659, 560, 421, 378, 326, 269, 214, 177, 126, 99, 102, 50, 60, 38, 37, 29, 18, 13, 8, 5, 4, 3, 3, 1, 2, 2]   # characters by times asked, 137 … 269

times through: 204.01 · times asked, on average: 204.01
between the ends: from 137 to 269


### ✋ Your turn

How many steps would it take for each character to be asked about 100 times on average? Use the same arithmetic as
`times_through`, turned round, and put the number in `answer`.

In [17]:
answer = None   # TODO: 100 times the text's length, over 64 × 32 answers a step

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 100 * n / (64 * 32)) < 1, '100 times through: 100 * n / (64 * 32) steps'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = 100 * n / (64 * 32)
print(f"about {answer:,.0f} steps, and times_through says {times_through(n, steps=round(answer)):.2f}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 100 * n / (64 * 32)) < 1, '100 times through: 100 * n / (64 * 32) steps'
    print("✓ right:", answer)

about 49,016 steps, and times_through says 100.00
✓ right: 49016.30859375


Every step draws 64 places from one seeded stream: each batch is fresh, yet the whole run can be drawn again, draw for draw.
Over 100,000 steps its windows ask about every character but 2, about 204 times each.

Next: **T2 · the loss**, where those 2,048 right answers a step become one number.